In [1]:
import sys
import os
import pickle
import numpy as np
import pandas as pd
import time

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, project_root)

from src.preprocessing import apply_smote, get_class_weights
from src.evaluation import compute_metrics, print_per_class_report, save_results

# Load all preprocessed data
processed_dir = os.path.join(project_root, "data", "processed")

X_train_full = pickle.load(open(os.path.join(processed_dir, "X_train_scaled.pkl"), "rb"))
X_test_full = pickle.load(open(os.path.join(processed_dir, "X_test_scaled.pkl"), "rb"))
y_train = pickle.load(open(os.path.join(processed_dir, "y_train.pkl"), "rb"))
y_test = pickle.load(open(os.path.join(processed_dir, "y_test.pkl"), "rb"))
target_encoder = pickle.load(open(os.path.join(processed_dir, "target_encoder.pkl"), "rb"))

# Load feature subsets
X_train_top10 = pickle.load(open(os.path.join(processed_dir, "X_train_top10.pkl"), "rb"))
X_test_top10 = pickle.load(open(os.path.join(processed_dir, "X_test_top10.pkl"), "rb"))
X_train_top20 = pickle.load(open(os.path.join(processed_dir, "X_train_top20.pkl"), "rb"))
X_test_top20 = pickle.load(open(os.path.join(processed_dir, "X_test_top20.pkl"), "rb"))
X_train_top30 = pickle.load(open(os.path.join(processed_dir, "X_train_top30.pkl"), "rb"))
X_test_top30 = pickle.load(open(os.path.join(processed_dir, "X_test_top30.pkl"), "rb"))
X_train_pca = pickle.load(open(os.path.join(processed_dir, "X_train_pca.pkl"), "rb"))
X_test_pca = pickle.load(open(os.path.join(processed_dir, "X_test_pca.pkl"), "rb"))

# Organize into a dictionary for easy looping
feature_sets = {
    "full-83": (X_train_full, X_test_full),
    "top-30": (X_train_top30, X_test_top30),
    "top-20": (X_train_top20, X_test_top20),
    "top-10": (X_train_top10, X_test_top10),
    "pca": (X_train_pca, X_test_pca),
}

# Create output directories
os.makedirs(os.path.join(project_root, "results"), exist_ok=True)
os.makedirs(os.path.join(project_root, "models"), exist_ok=True)

print("All data: ")
for name, (X_tr, X_te) in feature_sets.items():
    print(f"  {name}: train {X_tr.shape}, test {X_te.shape}")

All data: 
  full-83: train (98493, 83), test (24624, 83)
  top-30: train (98493, 30), test (24624, 30)
  top-20: train (98493, 20), test (24624, 20)
  top-10: train (98493, 10), test (24624, 10)
  pca: train (98493, 26), test (24624, 26)


In [2]:
# SVM Strategy:
# - RBF kernel (SVC) for baseline & cost-sensitive — 98K samples, fast enough
# - SGDClassifier with hinge loss for SMOTE — handles 908K samples in seconds
#
# SGDClassifier(loss='hinge') IS a linear SVM, just trained with SGD instead of
# coordinate descent. Same result, much faster on large data.

from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier

RANDOM_STATE = 42

def run_svm_experiment(X_train, y_train, X_test, y_test, 
                       feature_set_name, imbalance_strategy,
                       target_encoder=None):

    model_name = f"SVM | {feature_set_name} | {imbalance_strategy}"
    print(f"\n== Experiment: {model_name} ==")
    
    # Step 1: Apply imbalance strategy
    if imbalance_strategy == "smote":
        print("Applying SMOTE to training data...")
        X_train_exp, y_train_exp = apply_smote(X_train, y_train)
    else:
        X_train_exp, y_train_exp = X_train, y_train
    
    # Step 2: Choose SVM variant
    if imbalance_strategy == "smote":
        # SGD with hinge loss = linear SVM, handles 908K samples fast
        model = SGDClassifier(
            loss="hinge",
            max_iter=1000,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        kernel_used = "Linear-SGD"
    elif imbalance_strategy == "cost-sensitive":
        model = SVC(
            kernel="rbf",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            cache_size=1000
        )
        kernel_used = "RBF"
    else:
        model = SVC(
            kernel="rbf",
            random_state=RANDOM_STATE,
            cache_size=1000
        )
        kernel_used = "RBF"
    
    # Step 3: Train
    print(f"Kernel: {kernel_used} | Samples: {X_train_exp.shape[0]:,} | Features: {X_train_exp.shape[1]}")
    start_time = time.time()
    model.fit(X_train_exp, y_train_exp)
    train_time = time.time() - start_time
    print(f"Training time: {train_time:.1f}s")
    
    # Step 4: Predict & evaluate
    y_pred = model.predict(X_test)
    results = compute_metrics(y_test, y_pred, target_encoder=target_encoder)
    
    # Add metadata
    results["feature_set"] = feature_set_name
    results["imbalance_strategy"] = imbalance_strategy
    results["training_time"] = train_time
    results["kernel"] = kernel_used
    results["model_name"] = model_name
    
    # Step 5: Save results
    save_results(results, f"svm_{feature_set_name}_{imbalance_strategy}",
                 save_dir=os.path.join(project_root, "results"))
    
    # Step 6: Save model
    model_path = os.path.join(project_root, "models", f"svm_{feature_set_name}_{imbalance_strategy}.pkl")
    pickle.dump(model, open(model_path, "wb"))
    print(f"Model saved: {model_path}")
    
    return results

print("SVM experiment runner ready!")
print("Strategy: RBF for baseline & cost-sensitive, SGD-SVM for SMOTE")

SVM experiment runner ready!
Strategy: RBF for baseline & cost-sensitive, SGD-SVM for SMOTE


In [3]:
# --- SVM BASELINE EXPERIMENTS ---
# No imbalance handling — raw imbalanced data
# Uses RBF kernel (non-linear boundaries)
# This shows how SVM performs "out of the box" on each feature set
# Expect: good accuracy but lower macro F1 due to rare class misses

baseline_results = []

for fs_name, (X_tr, X_te) in feature_sets.items():
    result = run_svm_experiment(
        X_train=X_tr,
        y_train=y_train,
        X_test=X_te,
        y_test=y_test,
        feature_set_name=fs_name,
        imbalance_strategy="baseline",
        target_encoder=target_encoder
    )
    baseline_results.append(result)

# Summary
print(f"\n{'='*60}")
print("  SVM BASELINE SUMMARY (RBF kernel, no imbalance handling)")
print(f"{'='*60}")
print(f"{'Feature Set':<15} {'Macro F1':>10} {'Accuracy':>10} {'Time(s)':>10}")
print("-" * 47)
for r in baseline_results:
    print(f"{r['feature_set']:<15} {r['macro_f1']:>10.4f} {r['accuracy']:>10.4f} {r['training_time']:>9.1f}")


== Experiment: SVM | full-83 | baseline ==
Kernel: RBF | Samples: 98,493 | Features: 83
Training time: 4.2s

  Macro F1 (PRIMARY):  0.9402
  Accuracy:            0.9940
  Macro Precision:     0.9755
  Macro Recall:        0.9166

  Saved report to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_full-83_baseline_report.csv
  Saved summary to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_full-83_baseline_summary.csv
Model saved: /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/models/svm_full-83_baseline.pkl

== Experiment: SVM | top-30 | baseline ==
Kernel: RBF | Samples: 98,493 | Features: 30
Training time: 3.4s

  Macro F1 (PRIMARY):  0.9380
  Accuracy:            0.9940
  Macro Precision:     0.9730
  Macro Recall:        0.9144

  Saved report to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_top-30_baseline_report.csv
  S

In [4]:
# --- SVM SMOTE EXPERIMENTS ---
# SMOTE oversamples rare classes → training set grows from 98K to ~908K
# Using LinearSVC here because RBF on 908K samples would take hours
# Expect: recall on rare classes should improve, macro F1 may go up or stay similar

smote_results = []

for fs_name, (X_tr, X_te) in feature_sets.items():
    result = run_svm_experiment(
        X_train=X_tr,
        y_train=y_train,
        X_test=X_te,
        y_test=y_test,
        feature_set_name=fs_name,
        imbalance_strategy="smote",
        target_encoder=target_encoder
    )
    smote_results.append(result)

print(f"\n{'='*60}")
print("  SVM SMOTE SUMMARY (LinearSVC, SMOTE-resampled data)")
print(f"{'='*60}")
print(f"{'Feature Set':<15} {'Macro F1':>10} {'Accuracy':>10} {'Time(s)':>10}")
print("-" * 47)
for r in smote_results:
    print(f"{r['feature_set']:<15} {r['macro_f1']:>10.4f} {r['accuracy']:>10.4f} {r['training_time']:>9.1f}")


== Experiment: SVM | full-83 | smote ==
Applying SMOTE to training data...
  Before SMOTE: 98,493 samples


/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


  After SMOTE:  908,724 samples
Kernel: Linear-SGD | Samples: 908,724 | Features: 83
Training time: 5.7s

  Macro F1 (PRIMARY):  0.8185
  Accuracy:            0.9795
  Macro Precision:     0.7856
  Macro Recall:        0.9529

  Saved report to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_full-83_smote_report.csv
  Saved summary to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_full-83_smote_summary.csv
Model saved: /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/models/svm_full-83_smote.pkl

== Experiment: SVM | top-30 | smote ==
Applying SMOTE to training data...
  Before SMOTE: 98,493 samples


/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


  After SMOTE:  908,724 samples
Kernel: Linear-SGD | Samples: 908,724 | Features: 30
Training time: 1.7s

  Macro F1 (PRIMARY):  0.7556
  Accuracy:            0.9602
  Macro Precision:     0.7384
  Macro Recall:        0.9194

  Saved report to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_top-30_smote_report.csv
  Saved summary to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_top-30_smote_summary.csv
Model saved: /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/models/svm_top-30_smote.pkl

== Experiment: SVM | top-20 | smote ==
Applying SMOTE to training data...
  Before SMOTE: 98,493 samples


...

/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Training time: 1.1s

  Macro F1 (PRIMARY):  0.6246
  Accuracy:            0.9154
  Macro Precision:     0.6461
  Macro Recall:        0.8026

  Saved report to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_top-10_smote_report.csv
  Saved summary to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_top-10_smote_summary.csv
Model saved: /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/models/svm_top-10_smote.pkl

== Experiment: SVM | pca | smote ==
Applying SMOTE to training data...
  Before SMOTE: 98,493 samples


/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


  After SMOTE:  908,724 samples
Kernel: Linear-SGD | Samples: 908,724 | Features: 26
Training time: 2.7s

  Macro F1 (PRIMARY):  0.7147
  Accuracy:            0.9528
  Macro Precision:     0.6991
  Macro Recall:        0.9008

  Saved report to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_pca_smote_report.csv
  Saved summary to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_pca_smote_summary.csv
Model saved: /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/models/svm_pca_smote.pkl

  SVM SMOTE SUMMARY (LinearSVC, SMOTE-resampled data)
Feature Set       Macro F1   Accuracy    Time(s)
-----------------------------------------------
full-83             0.8185     0.9795       5.7
top-30              0.7556     0.9602       1.7
top-20              0.7101     0.9428       1.4
top-10              0.6246     0.9154       1.1
pca                 0.7147     0.9528       2.7


/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/piyushdaga/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [5]:
# --- SVM COST-SENSITIVE EXPERIMENTS ---
# RBF kernel with class_weight="balanced" — penalizes rare class errors more
# Same speed as baseline since no extra data is created (unlike SMOTE)
# This is the best of both worlds: non-linear boundaries + imbalance awareness
# Expect: should be close to or better than baseline

costsensitive_results = []

for fs_name, (X_tr, X_te) in feature_sets.items():
    result = run_svm_experiment(
        X_train=X_tr,
        y_train=y_train,
        X_test=X_te,
        y_test=y_test,
        feature_set_name=fs_name,
        imbalance_strategy="cost-sensitive",
        target_encoder=target_encoder
    )
    costsensitive_results.append(result)

print(f"\n{'='*60}")
print("  SVM COST-SENSITIVE SUMMARY (RBF kernel, balanced weights)")
print(f"{'='*60}")
print(f"{'Feature Set':<15} {'Macro F1':>10} {'Accuracy':>10} {'Time(s)':>10}")
print("-" * 47)
for r in costsensitive_results:
    print(f"{r['feature_set']:<15} {r['macro_f1']:>10.4f} {r['accuracy']:>10.4f} {r['training_time']:>9.1f}")


== Experiment: SVM | full-83 | cost-sensitive ==
Kernel: RBF | Samples: 98,493 | Features: 83
Training time: 6.6s

  Macro F1 (PRIMARY):  0.8380
  Accuracy:            0.9921
  Macro Precision:     0.8138
  Macro Recall:        0.9585

  Saved report to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_full-83_cost-sensitive_report.csv
  Saved summary to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_full-83_cost-sensitive_summary.csv
Model saved: /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/models/svm_full-83_cost-sensitive.pkl

== Experiment: SVM | top-30 | cost-sensitive ==
Kernel: RBF | Samples: 98,493 | Features: 30
Training time: 3.9s

  Macro F1 (PRIMARY):  0.8722
  Accuracy:            0.9918
  Macro Precision:     0.8433
  Macro Recall:        0.9615

  Saved report to /Users/piyushdaga/Desktop/ML/ML-Poject/CS-6140-Project-IoT-Intrusion-Detection/results/svm_

In [6]:
# --- CHECK RARE CLASS PERFORMANCE ACROSS ALL SVM EXPERIMENTS ---
# The real question: can our model detect NMAP_FIN_SCAN (6 test samples)
# and Metasploit_Brute_Force_SSH (7 test samples)?

import os

results_dir = os.path.join(project_root, "results")
rare_classes = ["Metasploit_Brute_Force_SSH", "NMAP_FIN_SCAN", "Wipro_bulb", "DDOS_Slowloris"]

print(f"{'Experiment':<35} {'Metric':<12} {'Metasploit':>12} {'FIN_SCAN':>12} {'Wipro':>12} {'Slowloris':>12}")
print("-" * 97)

for f in sorted(os.listdir(results_dir)):
    if f.startswith("svm_") and f.endswith("_report.csv"):
        name = f.replace("_report.csv", "")
        df = pd.read_csv(os.path.join(results_dir, f), index_col=0)
        
        # Get recall for rare classes
        recalls = []
        f1s = []
        for cls in rare_classes:
            if cls in df.index:
                recalls.append(df.loc[cls, "recall"])
                f1s.append(df.loc[cls, "f1-score"])
            else:
                recalls.append(0.0)
                f1s.append(0.0)
        
        print(f"{name:<35} {'Recall':<12} {recalls[0]:>12.3f} {recalls[1]:>12.3f} {recalls[2]:>12.3f} {recalls[3]:>12.3f}")
        print(f"{'':<35} {'F1':<12} {f1s[0]:>12.3f} {f1s[1]:>12.3f} {f1s[2]:>12.3f} {f1s[3]:>12.3f}")

Experiment                          Metric         Metasploit     FIN_SCAN        Wipro    Slowloris
-------------------------------------------------------------------------------------------------
svm_full-83_baseline                Recall              0.857        0.833        0.588        0.804
                                    F1                  0.923        0.833        0.741        0.891
svm_full-83_cost-sensitive          Recall              1.000        0.833        0.843        0.991
                                    F1                  0.246        0.256        0.804        0.865
svm_full-83_smote                   Recall              1.000        0.833        0.980        0.991
                                    F1                  0.165        0.385        0.752        0.831
svm_pca_baseline                    Recall              0.857        0.833        0.529        0.804
                                    F1                  0.923        0.714        0.692       